# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI and relabeling sequences. 

Notes:
* This file must be in the same folder as "utils.py"

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
import importlib
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

In [50]:
# Directory paths and input

# Input
browser = input("Browser (Firefox, Chrome, or Edge): ")
sleep_time = input("Seconds to wait in between clicks (recommended 5): ")
# locations = input("Locations (separate with commas and no spaces in between locations): ")
# start_date = input("Start date (format: MM-DD-YYYY): ")
# end_date = input("End date: (format: MM-DD-YYYY): ")
# serotype = input("Serotype (e.g. H5N1): ")
# serotypes = list(serotype)
# genotypes = input("Genotypes (separate with commas and no spaces in between genotypes): ")
# genotypes = genotypes.split(",")

# Dates and locations
locations = "Antarctica,North America,South America"
start_date = "11-01-2021"
end_date = "06-05-2026"
prev_end_date = "05-15-2026"
date_range = start_date + "--" + end_date
prev_date_range = start_date + "--" + prev_end_date

# Maintenance serotypes and genotypes
serotypes = ["H5N1"]
genotypes = ["B3.13"] #, "D1.1", "D1.3", "Not"]

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Downloads/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/"

home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
references = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/references/"

downloads_saved = home + "NCBI_Virus/downloads/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" 
prev_downloads_saved = home + "NCBI_Virus/downloads/" + prev_date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" 

andersen = home + "Andersen/avian-influenza/metadata/"
temp_files = home + "NCBI_Virus/temp/"
complete_files = home + "NCBI_Virus/complete/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

# All serotypes and genotypes
# serotype = ""
# genotypes_df = pd.read_excel("genotype_key.xlsx")
# genotypes = list(genotypes_df["Genotype"])


## Downloading Data

In [3]:
os.chdir(downloads)

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it

# Move downloaded files to saved downloads
for dirpath, dirs, files in os.walk(downloads_saved):
    if len(files) != 0:
        break 
    else: # If we don't have any downloaded files
        # Get files
        # open_ncbi_virus(browser, sleep_time, locations, start_date, end_date)

        # Re-try 
        for dirpath, dirs, files in os.walk(downloads):
            if len(files) > 0: # If we have any files that need to be moved
                for file in files:
                    file_name = os.path.join(dirpath, file)
                    destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                    try:
                        shutil.move(file_name, destination_path)
                    except:
                        print("Error moving file", file_name)
                        continue 
            break 
    break 

## De-Duplication

In [51]:
# Get metadata
os.chdir(downloads_saved)
metadata = pd.read_csv("sequences.csv")
print(len(metadata))

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.Assembly, as_index=False).size()
print(metadata_counts)
metadata_counted = metadata.merge(metadata_counts, on="Assembly")

# Only keep those with size >= 8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates
metadata_complete_segs = metadata_counted.drop_duplicates(subset="GenBank_Title", keep="last") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]
metadata_segments = metadata_complete_segs
metadata_segments

162482
              Assembly  size
0      GCA_038163845.1     8
1      GCA_038163995.1     8
2      GCA_038164135.1     8
3      GCA_038164315.1     8
4      GCA_038164335.1     8
...                ...   ...
18707  GCA_058020725.1     8
18708  GCA_058020755.1     8
18709  GCA_058020765.1     8
18710  GCA_058020775.1     8
18711  GCA_058020785.1     8

[18712 rows x 2 columns]


,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Host,Tissue_Specimen_Source,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size
0,OK205528.2,GenBank,GCA_038185045.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C.M., Parris,D.J., Kariithi,H....","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,1.0,2001,2021-11-01,ssRNA(-),8
1,OK205529.2,GenBank,GCA_038185045.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C.M., Parris,D.J., Kariithi,H....","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,1.0,2001,2021-11-01,ssRNA(-),8
2,OK205530.2,GenBank,GCA_038185045.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C.M., Parris,D.J., Kariithi,H....","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,1.0,2001,2021-11-01,ssRNA(-),8
3,OK205531.1,GenBank,GCA_038185045.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C.M., Parris,D.J., Kariithi,H....","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,1.0,2001,2021-11-01,ssRNA(-),8
4,OK205532.1,GenBank,GCA_038185045.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C.M., Parris,D.J., Kariithi,H....","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,1.0,2001,2021-11-01,ssRNA(-),8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149673,PP755920.1,GenBank,GCA_039463595.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Bos taurus,NaN,"Nguyen,T., Hutter,C., Markin,A., Thomas,M., La...","National Animal Disease Center, USDA-ARS",USA,NaN,2024-03-19,2024-05-03,ssRNA(-),8
149674,PP755922.1,GenBank,GCA_039463595.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Bos taurus,NaN,"Nguyen,T., Hutter,C., Markin,A., Thomas,M., La...","National Animal Disease Center, USDA-ARS",USA,NaN,2024-03-19,2024-05-03,ssRNA(-),8
149675,PP755924.1,GenBank,GCA_039463595.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Bos taurus,NaN,"Nguyen,T., Hutter,C., Markin,A., Thomas,M., La...","National Animal Disease Center, USDA-ARS",USA,NaN,2024-03-19,2024-05-03,ssRNA(-),8
149676,PP756066.1,GenBank,GCA_039464865.1,SRR28834851,SAMN41100354,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Bos taurus,mammalian milk,"Nguyen,T., Hutter,C., Markin,A., Thomas,M., La...","National Animal Disease Center, USDA-ARS",USA,NaN,2024-04-04,2024-05-03,ssRNA(-),8


## Add sequences to dataframe

In [52]:
# NCBI Virus Naming Convention:
# "Accession|GenBank_Title|Assembly|SRA Accession|BioSample|BioProject|Genotype|Isolate|Geo Location|Host|Collection Date"

# Get sequences and headers together
# headers = []
# isolates = []
# sras = []
# headers_seqs = {}

os.chdir(downloads_saved)

sequences_fasta = df_from_fasta("sequences.fasta") # Turn fasta into a dataframe

sequences_fasta["Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split(" |")[0].replace(">",""))

print(sequences_fasta["full_header"])

# Extract segment number so that we can add the correct sequences to the correct sample
# sequences_fasta["Segment"] = sequences_fasta["full_header"].apply(lambda x: int(re.search(r'segment (.?) ', x.split("|")[-2]).group(1)))

# Double-check the de-duplication
print(len(sequences_fasta)) 
# print(sequences_fasta.head())
print(len(metadata_segments))

# Add sequences to the dataframe
metadata_segments = pd.merge(metadata_segments, sequences_fasta, on="Accession") # , "Segment"])

0         >OK205528.2 |Influenza A virus (A/chicken/El S...
1         >OK205529.2 |Influenza A virus (A/chicken/El S...
2         >OK205530.2 |Influenza A virus (A/chicken/El S...
3         >OK205531.1 |Influenza A virus (A/chicken/El S...
4         >OK205532.1 |Influenza A virus (A/chicken/El S...
                                ...                        
162477    >PP755920.1 |Influenza A virus (A/cattle/Texas...
162478    >PP755922.1 |Influenza A virus (A/cattle/Texas...
162479    >PP755924.1 |Influenza A virus (A/cattle/Texas...
162480    >PP756066.1 |Influenza A virus (A/cattle/North...
162481    >PP756068.1 |Influenza A virus (A/cattle/North...
Name: full_header, Length: 162482, dtype: object
162482
148166


In [53]:
metadata_segments

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence
0,OK205528.2,GenBank,GCA_038185045.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C.M., Parris,D.J., Kariithi,H....","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,1.0,2001,2021-11-01,ssRNA(-),8,>OK205528.2 |Influenza A virus (A/chicken/El S...,TCAAATATATTCAATATGGAGAGAATAAAAGAACTAAGAGATCTAA...
1,OK205529.2,GenBank,GCA_038185045.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C.M., Parris,D.J., Kariithi,H....","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,1.0,2001,2021-11-01,ssRNA(-),8,>OK205529.2 |Influenza A virus (A/chicken/El S...,CAGGCAAACTATTTGAATGGATGTCAATCCGACTTTACTTTTCTTG...
2,OK205530.2,GenBank,GCA_038185045.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C.M., Parris,D.J., Kariithi,H....","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,1.0,2001,2021-11-01,ssRNA(-),8,>OK205530.2 |Influenza A virus (A/chicken/El S...,CAGGTACTGATCAAAAATGGAAGACTTCGTGCGACAGTGCTTCAAT...
3,OK205531.1,GenBank,GCA_038185045.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C.M., Parris,D.J., Kariithi,H....","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,1.0,2001,2021-11-01,ssRNA(-),8,>OK205531.1 |Influenza A virus (A/chicken/El S...,AGCAAAAGCAGGGGTATCAACCATCAAAATGAAAAGAATAGTGATT...
4,OK205532.1,GenBank,GCA_038185045.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C.M., Parris,D.J., Kariithi,H....","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,1.0,2001,2021-11-01,ssRNA(-),8,>OK205532.1 |Influenza A virus (A/chicken/El S...,AGCAAAAGCAGGGTAGATAATCACTCACCGAGTGACATTCACATCA...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148161,PP755920.1,GenBank,GCA_039463595.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Nguyen,T., Hutter,C., Markin,A., Thomas,M., La...","National Animal Disease Center, USDA-ARS",USA,NaN,2024-03-19,2024-05-03,ssRNA(-),8,>PP755920.1 |Influenza A virus (A/cattle/Texas...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...
148162,PP755922.1,GenBank,GCA_039463595.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Nguyen,T., Hutter,C., Markin,A., Thomas,M., La...","National Animal Disease Center, USDA-ARS",USA,NaN,2024-03-19,2024-05-03,ssRNA(-),8,>PP755922.1 |Influenza A virus (A/cattle/Texas...,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATTGTCGAGC...
148163,PP755924.1,GenBank,GCA_039463595.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Nguyen,T., Hutter,C., Markin,A., Thomas,M., La...","National Animal Disease Center, USDA-ARS",USA,NaN,2024-03-19,2024-05-03,ssRNA(-),8,>PP755924.1 |Influenza A virus (A/cattle/Texas...,ATGGAGAGAATAAAGGAACTGAGAGATCTAATGTCACAGTCTCGCA...
148164,PP756066.1,GenBank,GCA_039464865.1,SRR28834851,SAMN41100354,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Nguyen,T., Hutter,C., Markin,A., Thomas,M., La...","National Animal Disease Center, USDA-ARS",USA,NaN,2024-04-04,2024-05-03,ssRNA(-),8,>PP756066.1 |Influenza A virus (A/cattle/North...,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATTGTCGAGC...


## Find genotypes

(Using old genoflu results or Andersen Lab genoflu output) <br>
Old genoflu results should accumulate into one file to avoid having to genotype anything again.

### Find old genotypes

In [55]:
# Switch directory to previous week
os.chdir(prev_downloads_saved)

# Get both files from previous week
genoflu_output = pd.read_csv("output.tsv", delimiter="\t")
genoflu_results = pd.read_csv("results.tsv", delimiter="\t")

# Get old results from output.tsv
genoflu_old = pd.concat([genoflu_output, genoflu_results])
# genoflu_old = genoflu_old.rename(columns={"Strain":"Partial_Header"}) # So we can merge
print(genoflu_old)

# Switch back directory
os.chdir(downloads_saved)

# Save this output for the future
genoflu_old.to_csv("output.tsv", sep="\t", index=False)

                                                 Strain  \
0     Influenza_A_virus__Mexico__Estado_de_Mexico_CP...   
1     Influenza_A_virus__Mexico__Estado_de_Mexico_CP...   
2     Influenza_A_virus__Mexico__Ciudad_de_Mexico_CP...   
3     Influenza_A_virus__Mexico__Estado_de_Mexico_CP...   
4     Influenza_A_virus__Mexico__Michoacan_CPA_02011...   
...                                                 ...   
5473                                    GCA_051456995_1   
5474                                    GCA_046401875_1   
5475                                    GCA_051455955_1   
5476                                    GCA_056677775_1   
5477                                    GCA_055688435_1   

                                               Genotype  \
0     Not assigned: Only 3 segments >98.0% match fou...   
1     Not assigned: Only 3 segments >98.0% match fou...   
2     Not assigned: Only 3 segments >98.0% match fou...   
3     Not assigned: Only 3 segments >98.0% match fou...

In [56]:

metadata_segments["Partial_Header_temp"] = metadata_segments["Isolate"] # Get only isolate

for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]: # Forbidden punctuation
    metadata_segments["Partial_Header_temp"] = metadata_segments["Partial_Header_temp"].apply(lambda x: x.replace(c, "_") if x == x else x)

# Merge to get already-genotyped segments
genoflu_old["Partial_Header_temp"] = genoflu_old["Strain"].apply(lambda x: re.split(r'Influenza_A_virus__\D*__\D*_', x)[-1]) # Get only isolate 
genoflu_old["Partial_Header_temp"] = genoflu_old["Partial_Header_temp"].apply(lambda x: re.split(r'_H.N._20.{2}_.{2}_.{2}', x)[0]) # Get only isolate

metadata_segments_old = metadata_segments.merge(genoflu_old, how="inner", on="Partial_Header_temp") 

# Isolate not-already-genotyped segments
metadata_segments_new = metadata_segments.merge(genoflu_old, indicator=True, how='left', on="Partial_Header_temp").loc[lambda x : x['_merge']=='left_only'] 

print(len(metadata_segments_old))
print(len(metadata_segments_new))

128
148038


In [57]:
# Get genoflu results from Andersen and see if any fit

os.chdir(andersen)

genoflu_andersen = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_andersen = genoflu_andersen.rename(columns={"sample":"SRA_Accession"}) # So we can merge with old
# genoflu_andersen["SRA_Accession"] = genoflu_andersen["Strain"] # So we can merge with new

# Find those genotyped by Andersen via merge
metadata_segments_known_andersen = metadata_segments_new.merge(genoflu_andersen, how="inner", on="SRA_Accession")
print(metadata_segments_known_andersen)
# Rename genotype by Andersen to concatenate
metadata_segments_known_andersen["Genotype_y"] = metadata_segments_known_andersen["Genotype"]

# Keep all known genotypes
metadata_segments_known = pd.concat([metadata_segments_old, metadata_segments_known_andersen]) # , on="SRA_Accession", how="left") # Since we know both of these

print(metadata_segments_old)
print(metadata_segments_known_andersen)

        Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
0      PQ051119.1        GenBank  GCA_040889855.1   SRR29455647  SAMN41892119   
1      PQ051120.1        GenBank  GCA_040889855.1   SRR29455647  SAMN41892119   
2      PQ051121.1        GenBank  GCA_040889855.1   SRR29455647  SAMN41892119   
3      PQ051122.1        GenBank  GCA_040889855.1   SRR29455647  SAMN41892119   
4      PQ051123.1        GenBank  GCA_040889855.1   SRR29455647  SAMN41892119   
...           ...            ...              ...           ...           ...   
81283  PZ422028.1        GenBank  GCA_057730505.1   SRR38140017  SAMN57311643   
81284  PZ422029.1        GenBank  GCA_057730505.1   SRR38140017  SAMN57311643   
81285  PZ422030.1        GenBank  GCA_057730505.1   SRR38140017  SAMN57311643   
81286  PP756066.1        GenBank  GCA_039464865.1   SRR28834851  SAMN41100354   
81287  PP756068.1        GenBank  GCA_039464865.1   SRR28834851  SAMN41100354   

         BioProject      Or

### Create FASTA files of unknown genotypes 

In [58]:
metadata_segments_known

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Genotype Average Depth of Coverage List_x,_merge,date,File Name,Genotype,"Genotype List Used, >=98.0%_y",Genotype Sample Title List_y,Genotype Percent Match List_y,Genotype Mismatch List_y,Genotype Average Depth of Coverage List_y
0,OK205488.1,GenBank,GCA_039015945.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,OK205489.1,GenBank,GCA_039015945.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,OK205490.1,GenBank,GCA_039015945.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,OK205491.1,GenBank,GCA_039015945.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,OK205492.1,GenBank,GCA_039015945.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81283,PZ422028.1,GenBank,GCA_057730505.1,SRR38140017,SAMN57311643,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,left_only,2026-04-18_06-41-27,SRR38140017.fa,D1.8,"HA:ea3, NP:am1.1, NA:am4N1, PB1:ea3, PB2:am24,...","ea3:22-013001-001:HA, am1.1:22-007805-001:NP, ...","99.00%, 98.86%, 98.56%, 99.03%, 99.34%, 98.81%...","17, 17, 15, 22, 15, 10, 4, 18",Ran on FASTA - No Coverage Report
81284,PZ422029.1,GenBank,GCA_057730505.1,SRR38140017,SAMN57311643,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,left_only,2026-04-18_06-41-27,SRR38140017.fa,D1.8,"HA:ea3, NP:am1.1, NA:am4N1, PB1:ea3, PB2:am24,...","ea3:22-013001-001:HA, am1.1:22-007805-001:NP, ...","99.00%, 98.86%, 98.56%, 99.03%, 99.34%, 98.81%...","17, 17, 15, 22, 15, 10, 4, 18",Ran on FASTA - No Coverage Report
81285,PZ422030.1,GenBank,GCA_057730505.1,SRR38140017,SAMN57311643,PRJNA1207547,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,left_only,2026-04-18_06-41-27,SRR38140017.fa,D1.8,"HA:ea3, NP:am1.1, NA:am4N1, PB1:ea3, PB2:am24,...","ea3:22-013001-001:HA, am1.1:22-007805-001:NP, ...","99.00%, 98.86%, 98.56%, 99.03%, 99.34%, 98.81%...","17, 17, 15, 22, 15, 10, 4, 18",Ran on FASTA - No Coverage Report
81286,PP756066.1,GenBank,GCA_039464865.1,SRR28834851,SAMN41100354,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,left_only,2025-05-09_10-47-27,SRR28834851.fa,Not assigned: No Matching Genotypes,"PB1:am4, PA:ea1, NA:ea1, HA:ea1, NS:am1.1, MP:...","am4:23-001855-001:PB1, ea1:22-003707-003:PA, e...","99.55%, 99.22%, 98.94%, 98.63%, 99.28%, 98.78%...","4, 2, 15, 16, 6, 12, 2, 9",Ran on FASTA - No Coverage Report


In [62]:
# Create 8 fasta files per segment

# Get all the segments
metadata_segments_new["Partial_Header"] = metadata_segments_new["Assembly"].apply(lambda x: ">" + x if x == x else x) # .apply(lambda x: x.split("|")[-1].split(")")[0]) # Get only genbank name
metadata_segments_new = metadata_segments_new.dropna(subset="Partial_Header")

print(metadata_segments_new["Partial_Header"].values[0:5])

['>GCA_038185045.1' '>GCA_038185045.1' '>GCA_038185045.1'
 '>GCA_038185045.1' '>GCA_038185045.1']


In [ ]:


# Create list of dataframes
df_list = []
for partial_header in list(set(metadata_segments_new["Partial_Header"].values)): # Unique partial headers only
    # Get smaller dataframe
    df = metadata_segments_new[metadata_segments_new["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    # Make sure there are 8 segments
    if len(df) == 8:
        # Forbidden characters in headers
        for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]:
            df["Partial_Header"] = df["Partial_Header"].apply(lambda x: x.replace(c, "_"))
            
        df["full_header"] = df["Partial_Header"]
        df_list.append(df)

print(metadata_segments_new["Partial_Header"])

print(df_list[0]["full_header"].values[:10])

# Make fasta files
for df in df_list:
    segments = list(set(df["Segment"].apply(lambda x: int(x)).values))
    for segment in segments:
        one_row = df[df["Segment"] == segment]
        df_to_fasta(one_row, str(segment) + "_seg.fasta", temp_files)

['>GCA_038185045.1' '>GCA_038185045.1' '>GCA_038185045.1'
 '>GCA_038185045.1' '>GCA_038185045.1']


KeyboardInterrupt: 

### Re-Labeling Using GenoFlu

**STOP HERE AND USE GENOFLU TO FIND NEW GENOTYPES.** Then, make sure "results.tsv" is in the NCBI_Virus downloads directory. <br>

To activate genoflu conda environment in BioWulf:
```
source myconda
conda activate genoflu
```

To run GenoFLU-multi, change directories to Multi-GenoFLU directory:
``` 
cd GenoFLU-multi
```

And then call the python script:

``` 
python bin/genoflu-multi.py -f <FASTA_directory>
```

In [ ]:
# Ensure that user does the above
input("Use genoflu. Afterwards, press ENTER to continue.")

''

In [63]:
# Merging

os.chdir(downloads_saved)

# Read in genoflu results
output_genoflu = pd.read_csv("results.tsv", delimiter="\t") # This also shows the IDs of the new sequences -- incorporate into sequences report

# Create partial headers to merge genoflu results with previously unknown segments
metadata_segments_new["Partial_Header_Merge"] = metadata_segments_new["Assembly"] 
for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]: # Forbidden punctuation
    metadata_segments_new["Partial_Header_Merge"] = metadata_segments_new["Partial_Header_Merge"].apply(lambda x: x.replace(c, "_") if x==x else x)

# Reformat so we can merge
output_genoflu["Partial_Header_Merge"] = output_genoflu["Strain"] #.apply(lambda x: re.split(r'Influenza_A_virus__\D*__\D*_', x)[-1])
# output_genoflu["Partial_Header_Merge"] = output_genoflu["Partial_Header_Merge"].apply(lambda x: re.split(r'_H.N._(20|19).{2}_.{2}_.{2}', x)[0])

# Merge
metadata_genoflu = metadata_segments_new.merge(output_genoflu, how="inner", on="Partial_Header_Merge") #, suffixes=('_left', '_right')) 

# Fill the rest of the 8 segments with the same genotype
metadata_genoflu = metadata_genoflu.ffill(limit_area="inside", limit=7)



print(metadata_genoflu)


C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_26996\2649279360.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  metadata_genoflu = metadata_genoflu.ffill(limit_area="inside", limit=7)


         Accession GenBank_RefSeq         Assembly SRA_Accession  \
0       PQ051119.1        GenBank  GCA_040889855.1   SRR29455647   
1       PQ051120.1        GenBank  GCA_040889855.1   SRR29455647   
2       PQ051121.1        GenBank  GCA_040889855.1   SRR29455647   
3       PQ051122.1        GenBank  GCA_040889855.1   SRR29455647   
4       PQ051123.1        GenBank  GCA_040889855.1   SRR29455647   
...            ...            ...              ...           ...   
104123  PP761582.1        GenBank  GCA_040104305.1           NaN   
104124  PP761583.1        GenBank  GCA_040104305.1           NaN   
104125  PP761584.1        GenBank  GCA_040104305.1           NaN   
104126  PP761585.1        GenBank  GCA_040104305.1           NaN   
104127  PP761586.1        GenBank  GCA_040104305.1           NaN   

           BioSample    BioProject      Organism_Name  \
0       SAMN41892119  PRJNA1102327  Influenza A virus   
1       SAMN41892119  PRJNA1102327  Influenza A virus   
2       SAMN

### Concatenate with known genotypes

In [64]:
# Rename columns so we can concatenate
print(metadata_segments_known.columns)

metadata_segments_known = metadata_segments_known.reset_index(drop=True)
print(metadata_segments_known)

print(metadata_genoflu.columns)
metadata_segments_known["Genotype_official"] = metadata_segments_known["Genotype_y"]
metadata_segments_known["Serotype"] = metadata_segments_known["Genotype_x"]
metadata_genoflu["Genotype_official"] = metadata_genoflu["Genotype"]
metadata_genoflu["Serotype"] = metadata_genoflu["Genotype_x"]

metadata_genoflu = metadata_genoflu.reset_index(drop=True)
print(metadata_genoflu)

# Concatenation
metadata_genoflu_concat = pd.concat([metadata_segments_known, metadata_genoflu])

metadata_genoflu_concat

Index(['Accession', 'GenBank_RefSeq', 'Assembly', 'SRA_Accession', 'BioSample',
       'BioProject', 'Organism_Name', 'Species', 'Genus', 'Family',
       'Genotype_x', 'Isolate', 'Segment', 'GenBank_Title', 'Length',
       'Nuc_Completeness', 'Geo_Location', 'Country', 'USA', 'Host',
       'Tissue_Specimen_Source', 'Submitters', 'Organization', 'Org_location',
       'Publications', 'Collection_Date', 'Release_Date', 'Molecule_type',
       'size', 'full_header', 'sequence', 'Partial_Header_temp', 'Strain',
       'Genotype_y', 'Genotype List Used, >=98.0%',
       'Genotype Sample Title List', 'Genotype Percent Match List',
       'Genotype Mismatch List', 'Genotype Average Depth of Coverage List',
       'Date run', 'Genotype List Used, >=98.0%_x',
       'Genotype Sample Title List_x', 'Genotype Percent Match List_x',
       'Genotype Mismatch List_x', 'Genotype Average Depth of Coverage List_x',
       '_merge', 'date', 'File Name', 'Genotype',
       'Genotype List Used, >=98.0

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Genotype Mismatch List_y,Genotype Average Depth of Coverage List_y,Genotype_official,Serotype,Strain_x,Date run_x,Partial_Header_Merge,Partial_Header,Strain_y,Date run_y
0,OK205488.1,GenBank,GCA_039015945.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,Not assigned: Only 0 segments >98.0% match fou...,H5N2,NaN,NaN,NaN,NaN,NaN,NaN
1,OK205489.1,GenBank,GCA_039015945.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,Not assigned: Only 0 segments >98.0% match fou...,H5N2,NaN,NaN,NaN,NaN,NaN,NaN
2,OK205490.1,GenBank,GCA_039015945.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,Not assigned: Only 0 segments >98.0% match fou...,H5N2,NaN,NaN,NaN,NaN,NaN,NaN
3,OK205491.1,GenBank,GCA_039015945.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,Not assigned: Only 0 segments >98.0% match fou...,H5N2,NaN,NaN,NaN,NaN,NaN,NaN
4,OK205492.1,GenBank,GCA_039015945.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,Not assigned: Only 0 segments >98.0% match fou...,H5N2,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104123,PP761582.1,GenBank,GCA_040104305.1,NaN,NaN,PRJNA885497,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"4, 8, 13, 2, 5, 4, 5, 8",Ran on FASTA - No Coverage Report,C2.1,H5N1,NaN,NaN,GCA_040104305_1,>GCA_040104305.1,GCA_040104305_1,2026-06-10_12-23-04
104124,PP761583.1,GenBank,GCA_040104305.1,NaN,NaN,PRJNA885497,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"4, 8, 13, 2, 5, 4, 5, 8",Ran on FASTA - No Coverage Report,C2.1,H5N1,NaN,NaN,GCA_040104305_1,>GCA_040104305.1,GCA_040104305_1,2026-06-10_12-23-04
104125,PP761584.1,GenBank,GCA_040104305.1,NaN,NaN,PRJNA885497,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"4, 8, 13, 2, 5, 4, 5, 8",Ran on FASTA - No Coverage Report,C2.1,H5N1,NaN,NaN,GCA_040104305_1,>GCA_040104305.1,GCA_040104305_1,2026-06-10_12-23-04
104126,PP761585.1,GenBank,GCA_040104305.1,NaN,NaN,PRJNA885497,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"4, 8, 13, 2, 5, 4, 5, 8",Ran on FASTA - No Coverage Report,C2.1,H5N1,NaN,NaN,GCA_040104305_1,>GCA_040104305.1,GCA_040104305_1,2026-06-10_12-23-04


In [65]:
# Cut down to only columns we want
metadata_genoflu_concat = metadata_genoflu_concat[["Accession", "Assembly", "GenBank_Title", "Host", "Collection_Date", "SRA_Accession", "Isolate", "Genotype_official", "Geo_Location", "full_header", "sequence", "Serotype", "Segment", "File Name", "Partial_Header"]] #, "Strain"]]

# Get genbank strain name
metadata_genoflu_concat["genbank_name"] = metadata_genoflu_concat["GenBank_Title"].apply(lambda x: x.split("(")[1] if x == x else x)
metadata_genoflu_concat["Host"] = metadata_genoflu_concat["genbank_name"].apply(lambda x: x.split("/")[1] if x == x else x)

metadata_genoflu_concat = metadata_genoflu_concat.dropna(subset="genbank_name") # [metadata_genoflu["Genotype"]  == "B3.13"]

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_26996\4267596827.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_genoflu_concat["genbank_name"] = metadata_genoflu_concat["GenBank_Title"].apply(lambda x: x.split("(")[1] if x == x else x)
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_26996\4267596827.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_genoflu_concat["Host"] = metadata_genoflu_concat["genbank_name"].apply(lambda x: x.split("/")[1] if x == x else x)

In [66]:
metadata_genoflu_concat

# Make sure we only have the serotype(s) we want
for serotype in serotypes:
    metadata_genoflu = metadata_genoflu[metadata_genoflu["Serotype"] == serotype]

## Relabeling Sequences

We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

In [67]:
# Re-re-name so that we don't break following code

metadata_genoflu = metadata_genoflu_concat

After running the below code, **STOP TO CHECK** if any new animals appear

In [68]:
# Animals 

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genoflu)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['great-tailed grackle', 'american wigeon', 'vulture', 'black swan', 'great egret', 'skunk', 'fregata magnificens', 'brown skua', 'common grackle', 'turkey vulture', 'harbor seal', 'emu', 'sanderling', 'chicken', 'black scoter', 'northern gannet', 'flamingo', 'guinea fowl', 'pintail duck', 'tundra swan', 'osprey', 'goose', 'common rhea', 'common eiders', 'franklin gull', 'guanay cormorant', 'greater white-fronted goose', 'brown pelican', 'chukar partridge', 'dunlin', 'american green-winged teal', 'falcon', 'herring gull', 'procellaria aequinoctialis', 'silver pheasant', 'bluebird', 'michigan', 'great blue heron', 'california', 'ohio', 'dendrocygna viduata', 'bos taurus', 'horned grebe', 'snow goose', 'snowy owl', 'gadwell', 'western sandpiper', 'long-eared owl', 'arctic tern', 'blue jay', 'thalasseus maximus', 'waterfowl', 'american robin', 'bluejay', 'ermine', 'serval', 'red fox', 'mute swan', 'south polar skua', 'cattle', 'buteogallus urubitinga', 'common eider', 'black turnstone', '

In [ ]:
input("Check animals output. Afterwards, press ENTER to continue.")

''

In [69]:
# Re-label sequences with no assigned genotype as "Unassigned"

metadata_genoflu["Genotype_official"] = metadata_genoflu["Genotype_official"].apply(lambda x: "Not" if "Not assigned" in str(x) else x) # Unassigned segments are labeled "Unassigned"
metadata_genoflu["Genotype"] = metadata_genoflu["Genotype_official"] # Rename column again to not break old code

In [70]:
metadata_genoflu

,Accession,Assembly,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype_official,Geo_Location,full_header,sequence,Serotype,Segment,File Name,Partial_Header,genbank_name,Genotype
0,OK205488.1,GCA_039015945.1,Influenza A virus (A/chicken/Jalisco/716-2/201...,chicken,2017-07-04,NaN,716-2,Not,Mexico: Jalisco,>OK205488.1 |Influenza A virus (A/chicken/Jali...,TCAAATATATTCAATATGGAGAGAATAAAAGAACTAAGAGATCTAA...,H5N2,1,NaN,NaN,A/chicken/Jalisco/716-2/2017,Not
1,OK205489.1,GCA_039015945.1,Influenza A virus (A/chicken/Jalisco/716-2/201...,chicken,2017-07-04,NaN,716-2,Not,Mexico: Jalisco,>OK205489.1 |Influenza A virus (A/chicken/Jali...,CAAACTATTTGAATGGATGTCAATCCGACTCTACTTTTCTTGAAAG...,H5N2,2,NaN,NaN,A/chicken/Jalisco/716-2/2017,Not
2,OK205490.1,GCA_039015945.1,Influenza A virus (A/chicken/Jalisco/716-2/201...,chicken,2017-07-04,NaN,716-2,Not,Mexico: Jalisco,>OK205490.1 |Influenza A virus (A/chicken/Jali...,GCAGGTACTGATTCAAAATGGAAGACTTTGTGCGACAGTGTTTCAA...,H5N2,3,NaN,NaN,A/chicken/Jalisco/716-2/2017,Not
3,OK205491.1,GCA_039015945.1,Influenza A virus (A/chicken/Jalisco/716-2/201...,chicken,2017-07-04,NaN,716-2,Not,Mexico: Jalisco,>OK205491.1 |Influenza A virus (A/chicken/Jali...,TTCAACTGTCAAAATGGAAAGGATAGTAATCGCCTTTGCAATAATC...,H5N2,4,NaN,NaN,A/chicken/Jalisco/716-2/2017,Not
4,OK205492.1,GCA_039015945.1,Influenza A virus (A/chicken/Jalisco/716-2/201...,chicken,2017-07-04,NaN,716-2,Not,Mexico: Jalisco,>OK205492.1 |Influenza A virus (A/chicken/Jali...,GTAGATAATCACTCACTGAGTGACATTTACATCATGGCGTCTCAAG...,H5N2,5,NaN,NaN,A/chicken/Jalisco/716-2/2017,Not
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104123,PP761582.1,GCA_040104305.1,Influenza A virus (A/wood duck/South Carolina/...,wood duck,2023-12-20,NaN,A/wood duck/South Carolina/W24-007C/2023,C2.1,USA,>PP761582.1 |Influenza A virus (A/wood duck/So...,ATGAATCCAAATCAAAAGATAACAACCATTGGATCAATCTGTATGG...,H5N1,6,NaN,>GCA_040104305.1,A/wood duck/South Carolina/W24-007C/2023,C2.1
104124,PP761583.1,GCA_040104305.1,Influenza A virus (A/wood duck/South Carolina/...,wood duck,2023-12-20,NaN,A/wood duck/South Carolina/W24-007C/2023,C2.1,USA,>PP761583.1 |Influenza A virus (A/wood duck/So...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAGATGGAGACTG...,H5N1,5,NaN,>GCA_040104305.1,A/wood duck/South Carolina/W24-007C/2023,C2.1
104125,PP761584.1,GCA_040104305.1,Influenza A virus (A/wood duck/South Carolina/...,wood duck,2023-12-20,NaN,A/wood duck/South Carolina/W24-007C/2023,C2.1,USA,>PP761584.1 |Influenza A virus (A/wood duck/So...,ATGGAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...,H5N1,4,NaN,>GCA_040104305.1,A/wood duck/South Carolina/W24-007C/2023,C2.1
104126,PP761585.1,GCA_040104305.1,Influenza A virus (A/wood duck/South Carolina/...,wood duck,2023-12-20,NaN,A/wood duck/South Carolina/W24-007C/2023,C2.1,USA,>PP761585.1 |Influenza A virus (A/wood duck/So...,ATGAGTCTTCTAACCGAGGTCGAAACGTACGTTCTCTCTATCGTCC...,H5N1,7,NaN,>GCA_040104305.1,A/wood duck/South Carolina/W24-007C/2023,C2.1


In [71]:
# Re-Labeling

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

# Fix animals
fix_animals_andersen(metadata_genoflu, animals_ref)

# Get the years
metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))

# Get geographic locations
metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location"].apply(lambda x: 
                                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                                        states_ref.loc[states_ref["Abbreviation"].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                                        + "-" + 
                                                                        x.split(" ")[-1]
                                                                        if states_ref["Abbreviation"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has the full state name (e.g. "Maryland")
                                                                        else states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                                        + "-" + 
                                                                        states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                                        if states_ref["State"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has neither the state abbreviation nor the full state name nor is "USA"
                                                                        else 
                                                                        x
                                                                        )

metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location_Abrv"].apply(lambda x: x.split("-")[0] if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] else x)


print(metadata_genoflu["Geo_Location_Abrv"])

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_26996\4197312387.py:22: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  if states_ref["Abbreviation"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_26996\4197312387.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  states_ref.loc[states_ref["Abbreviation"].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]


0         Mexico-Jalisco
1         Mexico-Jalisco
2         Mexico-Jalisco
3         Mexico-Jalisco
4         Mexico-Jalisco
               ...      
104123               USA
104124               USA
104125               USA
104126               USA
104127               USA
Name: Geo_Location_Abrv, Length: 185544, dtype: object


In [72]:
# If there is no SRA Accession, replace identifier with Assembly -- this part might be duplicating accessions :(
metadata_genoflu["Identifier"] = metadata_genoflu["Assembly"].apply(lambda x: x if metadata_genoflu[metadata_genoflu["Assembly"] == x]["SRA_Accession"].values[0] == "" else metadata_genoflu[metadata_genoflu["Assembly"] == x]["SRA_Accession"].values[0]) # np.where(metadata_genoflu['SRA_Accession'] != "", metadata_genoflu['SRA_Accession'], metadata_genoflu['Assembly'].apply(lambda x: x.split(".")[0]))
# # If there is no Assembly, replace identifier with Accession -- only for PB2
# metadata_genoflu["Identifier"] = np.where(metadata_genoflu["SRA_Accession_intermediate"] == "", metadata_genoflu["Accession"].apply(lambda x: metadata_genoflu[metadata_genoflu["Accession"] == x]["Accession"].values[0] if metadata_genoflu[metadata_genoflu["Accession"] == x].loc[:, "Segment"].values[0] == 1 else np.nan), metadata_genoflu["SRA_Accession_intermediate"])
# # Fill in other nans with PB2 Accession (interpolate, maximum of 7 other sequences)
# metadata_genoflu.loc[:,"Identifier"] = metadata_genoflu.loc[:, "Identifier"].ffill(limit=7, limit_area="inside")

print((metadata_genoflu[metadata_genoflu["Identifier"].str.contains("SRR")])) # metadata_genoflu[(metadata_genoflu["Identifier"].str.contains("GCA")) | 

         Accession         Assembly  \
128     PQ051119.1  GCA_040889855.1   
129     PQ051120.1  GCA_040889855.1   
130     PQ051121.1  GCA_040889855.1   
131     PQ051122.1  GCA_040889855.1   
132     PQ051123.1  GCA_040889855.1   
...            ...              ...   
104051  PZ422026.1  GCA_057730505.1   
104052  PZ422027.1  GCA_057730505.1   
104053  PZ422028.1  GCA_057730505.1   
104054  PZ422029.1  GCA_057730505.1   
104055  PZ422030.1  GCA_057730505.1   

                                            GenBank_Title        Host  \
128     Influenza A virus (A/Alpaca/ID/24-014328-008/2...      alpaca   
129     Influenza A virus (A/Alpaca/ID/24-014328-008/2...      alpaca   
130     Influenza A virus (A/Alpaca/ID/24-014328-008/2...      alpaca   
131     Influenza A virus (A/Alpaca/ID/24-014328-008/2...      alpaca   
132     Influenza A virus (A/Alpaca/ID/24-014328-008/2...      alpaca   
...                                                   ...         ...   
104051  Influenza A 

In [73]:

# Make new labels
names = ">" + metadata_genoflu["Identifier"].astype(str) + "|" + metadata_genoflu["genbank_name"] + "|" + metadata_genoflu["Serotype"] + "|" + metadata_genoflu["Geo_Location_Abrv"] + "|" + metadata_genoflu["Collection_Date"].astype(str) + "|" + metadata_genoflu["Host_Type"] + "|" + metadata_genoflu["Genotype"]

metadata_genoflu["Name"] = names



In [74]:
os.chdir(complete_files)

# De-duplicate

print(len(metadata_genoflu))
metadata_genoflu = metadata_genoflu.drop_duplicates(subset=["Isolate", "genbank_name", "Segment"], keep="last")
print(len(metadata_genoflu))

# Save metadata
metadata_genoflu.to_csv("NCBI_Virus_" + date_range + "_metadata.csv") # Make file for metadata


185544
118656


## Rename segments and make complete FASTA files

In [75]:
# Set up segments

# if len(genotypes) > 3: # If we're not doing maintenance only
#     genotypes.append("Not assigned") # Make sure unassigned genotypes are included
segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"} # Name segments 
metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].apply(lambda x: int(x)).map(segments)

# Separate into several dataframes based on genotype + segment
segment_genotype_dfs = []
for segment in segments.values():
    m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
    for genotype in list(set(m_g["Genotype"].values)):
        if genotype in genotypes:
            df = m_g[(m_g["Genotype"] == genotype)] 
            # df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
            segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)

B3.13_PB2
B3.13_PB1
B3.13_PA
B3.13_HA
B3.13_NP
B3.13_NA
B3.13_MP
B3.13_NS


In [77]:
# Create FASTA files

os.chdir(complete_files)

for df in segment_genotype_dfs:
    print(df)
    
    df = df.reset_index()
    if len(df["Genotype"].values[0]) > 0: # If we have assigned genotypes, including "Unassigned"
        file_name = df["Genotype"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
        output_file = open(complete_files + file_name, "w")

        for index, row in df.iterrows():
            name = df.loc[index, "Name"]
            name = name.replace(" ", "_")
            # names.append(name)
            sequence = df.loc[index, "sequence"]
            # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(sequence + "\n")
        
    output_file.close()

         Accession         Assembly  \
4528    PV070690.1  GCA_047521275.1   
4536    PV070698.1  GCA_047521295.1   
10768   PQ010976.1  GCA_042347135.1   
10776   PQ012072.1  GCA_042347435.1   
10784   PQ011176.1  GCA_040780605.1   
...            ...              ...   
103192  PV601972.1  GCA_050381395.1   
103200  PV601948.1  GCA_050381405.1   
103208  PV602004.1  GCA_050381415.1   
103216  PV602012.1  GCA_050381425.1   
103224  PV601980.1  GCA_050381435.1   

                                            GenBank_Title    Host  \
4528    Influenza A virus (A/Bobcat/WA/24-037410-006-o...  bobcat   
4536    Influenza A virus (A/Bobcat/WA/24-037410-013-o...  bobcat   
10768   Influenza A virus (A/Cat/New Mexico/24-010948-...     cat   
10776   Influenza A virus (A/Cat/Texas/24-009311-002-o...     cat   
10784   Influenza A virus (A/Cattle/Colorado/24-012225...  cattle   
...                                                   ...     ...   
103192  Influenza A virus (A/swine/Kansas/ExpPig